# Module 02 -- Greeks Intuition

**Author: Djellal Djouad** -- CrossVol Research | [crossvol.com](https://crossvol.com) | ORCID [0009-0002-4911-1118](https://orcid.org/0009-0002-4911-1118)

The Greeks are your dashboard. Delta tells you direction, gamma tells you how fast
that direction changes, theta is the rent you pay, and vega is your bet on uncertainty.
Every morning on the desk, the first thing you look at is your gamma profile -- because
that's what can blow up before lunch.

This notebook computes all four analytically from Black-Scholes-Merton and builds
3D surfaces so you can *see* how they behave across spot and time.

*License: MIT with Educational Use Clause -- see LICENSE. Not trading advice.*

**References:**
- *Beyond Greeks & Exotics* -- [Amazon](https://www.amazon.com/dp/B0H2RZGMY6)
- Djouad (2025), *Rethinking the Greeks* -- [doi:10.5281/zenodo.20509786](https://doi.org/10.5281/zenodo.20509786)


In [ ]:
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt


In [ ]:
# --- BSM Greeks (closed-form) ---
# Nothing fancy here. These are the textbook formulas, but I'm writing them out
# so you can trace every sign and factor yourself. Trust formulas you can derive.

def bsm_d1(S, K, T, r, sigma):
    """d1 from Black-Scholes. The backbone of everything."""
    return (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))

def bsm_d2(S, K, T, r, sigma):
    return bsm_d1(S, K, T, r, sigma) - sigma * np.sqrt(T)

def delta_call(S, K, T, r, sigma):
    return norm.cdf(bsm_d1(S, K, T, r, sigma))

def delta_put(S, K, T, r, sigma):
    return delta_call(S, K, T, r, sigma) - 1.0

def gamma(S, K, T, r, sigma):
    d1 = bsm_d1(S, K, T, r, sigma)
    return norm.pdf(d1) / (S * sigma * np.sqrt(T))

def theta_call(S, K, T, r, sigma):
    d1 = bsm_d1(S, K, T, r, sigma)
    d2 = bsm_d2(S, K, T, r, sigma)
    term1 = -(S * norm.pdf(d1) * sigma) / (2.0 * np.sqrt(T))
    term2 = -r * K * np.exp(-r * T) * norm.cdf(d2)
    return (term1 + term2) / 365.0  # per calendar day

def vega(S, K, T, r, sigma):
    d1 = bsm_d1(S, K, T, r, sigma)
    return S * norm.pdf(d1) * np.sqrt(T) / 100.0  # per 1 vol point


## Delta -- Your Directional Exposure

Delta ranges from 0 to 1 for calls, -1 to 0 for puts. ATM options sit around 0.50.
On the desk we think of delta as "equivalent shares" -- a 0.30 delta call on 100 shares
behaves like owning 30 shares. Simple, but people forget that delta *changes* as
spot moves. That's gamma.


In [ ]:
S_range = np.linspace(80, 120, 200)
K = 100
r = 0.05
sigma = 0.20

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for T, label in [(0.5, '6M'), (0.08, '1M'), (0.01, '~4 days')]:
    d = delta_call(S_range, K, T, r, sigma)
    axes[0].plot(S_range, d, label=f'T = {label}')
    dp = delta_put(S_range, K, T, r, sigma)
    axes[1].plot(S_range, dp, label=f'T = {label}')

axes[0].set_title('Call Delta')
axes[0].set_xlabel('Spot')
axes[0].axhline(0.5, color='grey', ls='--', lw=0.7)
axes[0].axvline(K, color='grey', ls='--', lw=0.7)
axes[0].legend()

axes[1].set_title('Put Delta')
axes[1].set_xlabel('Spot')
axes[1].axhline(-0.5, color='grey', ls='--', lw=0.7)
axes[1].axvline(K, color='grey', ls='--', lw=0.7)
axes[1].legend()

plt.tight_layout()
plt.show()


## Gamma -- The Rate of Change of Delta

Gamma is highest ATM and near expiry. This is the thing that keeps derivatives
traders up at night during expiration week. If you're short gamma near the pin
(spot right at the strike), your delta swings wildly and you're rehedging every
few minutes. I've seen people lose a full year of P&L on a single expiry Friday
because they sat on short gamma and the stock pinned their strike.

**Desk perspective:** Gamma near expiry is what kills you on pin risk. If you're
short 10,000 gammas with 2 hours to go and the stock is sitting on your strike,
every tick changes your delta by thousands of shares. You're basically flipping
a coin on direction. The smart move is to flatten or buy it back before it gets there.


In [ ]:
S_grid = np.linspace(85, 115, 100)
T_grid = np.linspace(0.01, 1.0, 100)
S_mesh, T_mesh = np.meshgrid(S_grid, T_grid)

G = gamma(S_mesh, K, T_mesh, r, sigma)

fig = plt.figure(figsize=(11, 7))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(S_mesh, T_mesh, G, cmap='inferno', alpha=0.85)
ax.set_xlabel('Spot')
ax.set_ylabel('Time to Expiry (years)')
ax.set_zlabel('Gamma')
ax.set_title('Gamma Surface -- notice the spike ATM near expiry')
ax.view_init(elev=25, azim=-50)
plt.tight_layout()
plt.show()


## Theta -- Time Decay

Theta is always negative for long options (you're paying rent for optionality).
ATM options decay fastest. Deep ITM or OTM options barely decay because they're
either certain or worthless -- there's nothing left to resolve.

One thing they don't teach in textbooks: theta accelerates into expiry for ATM
options. The last week is brutal. If you're long ATM weeklies and the stock goes
nowhere, you'll feel every hour tick by.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for T, label in [(1.0, '1Y'), (0.25, '3M'), (0.04, '~2 weeks')]:
    th = theta_call(S_range, K, T, r, sigma)
    ax.plot(S_range, th, label=f'T = {label}')

ax.set_title('Call Theta (per calendar day)')
ax.set_xlabel('Spot')
ax.set_ylabel('Theta')
ax.axvline(K, color='grey', ls='--', lw=0.7)
ax.legend()
plt.tight_layout()
plt.show()


## Vega -- Sensitivity to Volatility

Vega is highest ATM and *far* from expiry -- the opposite pattern from gamma.
This makes intuitive sense: a long-dated ATM option has the most "uncertainty"
left to resolve, so a change in implied vol has the biggest impact.

When I was running a vol book, vega was the main risk metric. Delta you can hedge.
Gamma you can manage with position sizing. But vega -- that's your core bet.
If you're long vega and vol collapses, there's no hedge that saves you.


In [ ]:
V = vega(S_mesh, K, T_mesh, r, sigma)

fig = plt.figure(figsize=(11, 7))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(S_mesh, T_mesh, V, cmap='viridis', alpha=0.85)
ax.set_xlabel('Spot')
ax.set_ylabel('Time to Expiry (years)')
ax.set_zlabel('Vega (per 1 vol pt)')
ax.set_title('Vega Surface -- peaks ATM, far from expiry')
ax.view_init(elev=25, azim=-50)
plt.tight_layout()
plt.show()


## All Greeks at a Glance

Let's plot all four for a single option as spot moves. This is the kind of
one-pager you'd have taped to your monitor as a junior.


In [ ]:
T_fixed = 0.25  # 3 months
S_plot = np.linspace(80, 120, 300)

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

axes[0, 0].plot(S_plot, delta_call(S_plot, K, T_fixed, r, sigma), color='steelblue')
axes[0, 0].set_title('Delta (Call)')
axes[0, 0].axvline(K, color='grey', ls='--', lw=0.7)

axes[0, 1].plot(S_plot, gamma(S_plot, K, T_fixed, r, sigma), color='firebrick')
axes[0, 1].set_title('Gamma')
axes[0, 1].axvline(K, color='grey', ls='--', lw=0.7)

axes[1, 0].plot(S_plot, theta_call(S_plot, K, T_fixed, r, sigma), color='darkorange')
axes[1, 0].set_title('Theta (per day)')
axes[1, 0].axvline(K, color='grey', ls='--', lw=0.7)

axes[1, 1].plot(S_plot, vega(S_plot, K, T_fixed, r, sigma), color='seagreen')
axes[1, 1].set_title('Vega (per 1 vol pt)')
axes[1, 1].axvline(K, color='grey', ls='--', lw=0.7)

for ax in axes.flat:
    ax.set_xlabel('Spot')
    ax.grid(True, alpha=0.3)

fig.suptitle(f'Greeks -- K={K}, T={T_fixed}Y, r={r}, sigma={sigma}', fontsize=13)
plt.tight_layout()
plt.show()


## Key Takeaways

1. **Gamma peaks ATM near expiry** -- this is pin risk. Respect it.
2. **Vega peaks ATM far from expiry** -- long-dated ATM options are vol bets.
3. **Theta is the cost of gamma** -- you can't have convexity for free.
4. **Delta is the simplest Greek** but don't forget it moves (that's gamma).

On a real desk, you manage gamma intraday and vega over weeks. Theta is the bleed
you accept in exchange for the convexity you want. Everything is a trade-off.

---

**Next:** [Module 03 -- FX Options (Garman-Kohlhagen)](03_fx_options.py)

*Djellal Djouad -- CrossVol Research -- 2026*
